# 03 — PlacementIQ Data Preprocessing & Validation Pipeline

This notebook covers end-to-end data preparation for PlacementIQ, ensuring strict leakage-safe splits, custom transformer pipelines with standard scaling and one-hot encoding, imputation verification, handle_unknown behavior check, and stratified k-fold splits preview.

## Section 1 — Setup
- Imports: pandas, numpy, matplotlib, seaborn, sklearn, joblib, sys, pathlib
- Add `src/` to `sys.path` to import from `preprocessing.py`
- Load dataset and drop target leakage column (`salary_package_lpa`)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib

# Add src to sys.path
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from preprocessing import build_preprocessor, NUMERICAL_FEATURES, CATEGORICAL_FEATURES

# Load dataset
raw_path = Path("../data/raw/student_placement_synthetic.csv").resolve()
df = pd.read_csv(raw_path)
print(f"Initial raw dataset shape: {df.shape}")

# Drop salary_package_lpa
if "salary_package_lpa" in df.columns:
    df = df.drop(columns=["salary_package_lpa"])
print(f"Shape after dropping leakage column: {df.shape}")

## Section 2 — Feature Selection Rationale
The dataset contains 16 core features after dropping `salary_package_lpa`:
- **Categorical (2)**: `branch`, `college_tier`
- **Numerical (14)**: `cgpa`, `backlogs`, `coding_skills`, `dsa_score`, `aptitude_score`, `communication_skills`, `ml_knowledge`, `system_design`, `internships`, `projects_count`, `certifications`, `hackathons`, `open_source_contributions`, `extracurriculars`

> **Leakage Exclusion**: `salary_package_lpa` is strictly an outcome variable that is only known *after* placement is secured (and is missing/zero or correlated conditionally only with placed candidates). Retaining it would lead to direct target leakage and an artificially perfect but useless model in production.

In [ ]:
final_features = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
print(f"Total features chosen: {len(final_features)}")
print("Numerical features:", NUMERICAL_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)

## Section 3 — Leakage Check
Verify that `salary_package_lpa` is completely absent from candidate model features.

In [ ]:
assert "salary_package_lpa" not in final_features, "Error: salary_package_lpa detected in features!"
assert "salary_package_lpa" not in df.columns, "Error: salary_package_lpa still present in DataFrame!"
print("Assertion passed: No leakage column detected in features or data.")

## Section 4 — Stratified Train/Test Split
Splitting the dataset into 80% train and 20% test sets, stratified across `placement_status` to maintain the ~68.5% : 31.5% class balance.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[final_features]
y = df["placement_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

train_prop = y_train.value_counts(normalize=True).to_dict()
test_prop = y_test.value_counts(normalize=True).to_dict()

print("Train class proportions:", train_prop)
print("Test class proportions: ", test_prop)
assert abs(train_prop[1] - test_prop[1]) < 0.005, "Class proportion mismatch exceeds tolerance!"
print("Stratification check passed successfully.")

## Section 5 — Preprocessing Pipeline Construction
Constructing a ColumnTransformer pipeline:
- **Numerical Pipeline**: `SimpleImputer(strategy='median')` + `StandardScaler()` (optional flag)
- **Categorical Pipeline**: `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')`

> **Critical Note**: The preprocessor must ONLY be fitted on `X_train` to prevent data leakage from the test distribution.

In [ ]:
preprocessor = build_preprocessor(scale_numerics=True)
display(preprocessor)

# Fit exclusively on X_train
preprocessor.fit(X_train)
print("Preprocessor successfully fitted on training partition.")

## Section 6 — Transformation Verification
Transforming `X_train` and `X_test` using the fitted preprocessor, examining dimensions, output arrays, and feature names.

In [ ]:
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print(f"Input feature count:  {X_train.shape[1]}")
print(f"Output feature count: {X_train_trans.shape[1]}")

cat_encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]
ohe_feature_names = list(cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES))
all_feature_names = NUMERICAL_FEATURES + ohe_feature_names

print(f"Final transformed feature names ({len(all_feature_names)}):", all_feature_names)
print("\nFirst 2 transformed training rows:")
print(X_train_trans[:2])

## Section 7 — Missing Value Imputation Demo
Testing pipeline robustness against missing values by injecting artificial NaNs.

In [ ]:
X_train_nan = X_train.copy()
X_train_nan.iloc[0, 0] = np.nan  # Inject NaN in numerical feature
X_train_nan.iloc[0, -1] = np.nan # Inject NaN in categorical feature

X_nan_trans = preprocessor.transform(X_train_nan)
nan_count = np.isnan(X_nan_trans).sum()
print(f"Total NaNs in output array after pipeline transform: {nan_count}")
assert nan_count == 0, "Imputation failed! NaNs present in output."

## Section 8 — OHE Unknown Category Handling
Verifying that unseen categories in production do not break inference thanks to `handle_unknown='ignore'`. All OHE columns for that feature should be 0.

In [ ]:
sample = X_train.iloc[[0]].copy()
sample["branch"] = "Unknown_Branch"
sample_trans = preprocessor.transform(sample)

branch_ohe_cols = [i for i, name in enumerate(all_feature_names) if name.startswith("branch_")]
branch_encoded_vals = sample_trans[0, branch_ohe_cols]
print("Transformed values for branch OHE columns:", branch_encoded_vals)
assert np.all(branch_encoded_vals == 0), "handle_unknown='ignore' failed to output all zeros!"
print("Unknown category handled safely: all one-hot encoded branch columns are 0.")

## Section 9 — Cross-Validation Preview
Running a 5-fold Stratified K-Fold split on the training set to verify stratification across each validation fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Cross-Validation Preview across {skf.get_n_splits()} folds:")
print("Fold | Train Size | Val Size | Val Placed% | Val Not-Placed%")
print("-" * 58)

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    y_val = y_train.iloc[val_idx]
    val_prop = y_val.value_counts(normalize=True)
    placed_pct = val_prop.get(1, 0) * 100
    not_placed_pct = val_prop.get(0, 0) * 100
    print(f"{fold:4d} | {len(train_idx):10d} | {len(val_idx):8d} | {placed_pct:10.2f}% | {not_placed_pct:14.2f}%")

## Section 10 — Save Artifacts
- Save `X_train`, `X_test`, `y_train`, `y_test` to `data/processed/`
- Save fitted preprocessor to `models/preprocessor.joblib`

In [ ]:
processed_dir = Path("../data/processed").resolve()
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)
print(f"Datasets successfully saved to {processed_dir}")

models_dir = Path("../models").resolve()
models_dir.mkdir(parents=True, exist_ok=True)

preprocessor_path = models_dir / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)
print(f"Fitted preprocessor saved to {preprocessor_path}")